# Reusable study demonstration

Generated participants only. Run all cells in a fresh kernel after installing the Python and R dependencies. This example demonstrates cohort definitions, matching and inference; it makes no clinical claims.

In [ ]:
from pathlib import Path
import os
from aou_studies.specs import StudySpec
from aou_studies.report_content import ReportSpec
from aou_studies.runner import StudyRun
from aou_studies.synthetic import synthetic_features
from aou_studies.provenance import environment_versions
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples/synthetic.yaml').exists())
if (ROOT / 'r-library-path.txt').exists():
    os.environ['AOU_R_LIBRARY'] = (ROOT / 'r-library-path.txt').read_text().strip()
environment_versions()


## Define the study
The YAML file controls the groups, matching and models. Dates are literal and fixed.

In [ ]:
study = StudySpec.load(ROOT / 'examples/synthetic.yaml')
run = StudyRun(study, ROOT / 'outputs/synthetic_notebook', synthetic=True)
run.use_synthetic(synthetic_features(study))

## Build comparable groups and match
The library validates participant uniqueness, group separation and every matched set.

In [ ]:
run.build_groups()
run.match()

## Inspect balance before inference
These are synthetic diagnostics. Before matching uses the full eligible matching pool; after matching uses equal total control weight per case.

In [ ]:
display(run.matched.balance)

In [ ]:
run.analyze()

## Prepare review tables
Real study tables require scientific and complete-bundle disclosure review inside Workbench.

In [ ]:
run.report(spec=ReportSpec.load(ROOT / 'examples/report-content.yaml'))

## Reuse the same library for a different study
This example changes the condition, controls, eligibility age range, matching variables, method and ratio.

In [ ]:
alternative = StudySpec.load(ROOT / 'examples/alternate_study/study.yaml')
other = StudyRun(alternative, ROOT / 'outputs/alternate_notebook', synthetic=True)
other.use_synthetic(synthetic_features(alternative))
other.build_groups()
other.match()
other.analyze()
other.report(spec=ReportSpec.load(ROOT / "examples/alternate_study/report-content.yaml"))

## Format saved results for a paper
`run.report()` already writes `review/tables.xlsx`. To use a journal layout, load the saved aggregate report and a separate layout file. This step makes no cloud query and fits no model.

`report-content.yaml` defines the summaries computed inside the workspace. `publication.yaml` independently declares required manuscript/supplement content. Layout changes may split or reorder tables; the coverage check flags missing content before export. Included suppressed or non-estimable results are accounted for, but still need scientific and disclosure review.

In [ ]:
from aou_studies.coverage import PublicationSpec, check_coverage
from aou_studies.reporting import Report
from aou_studies.excel import WorkbookSpec
saved = Report.load(run.directory / 'review')
layout = WorkbookSpec.load(ROOT / 'examples/report-layout.yaml')
requirements = PublicationSpec.load(ROOT / 'examples/publication.yaml')
coverage = check_coverage({'main': saved}, layout=layout, requirements=requirements)
display(coverage.table)
coverage.require_complete()
saved.write_excel(run.directory / 'manuscript-tables.xlsx', layout=layout, requirements=requirements)

alternate_report = Report.load(other.directory / "review")
alternate_layout = WorkbookSpec.load(ROOT / "examples/alternate_study/report-layout.yaml")
alternate_requirements = PublicationSpec.load(ROOT / "examples/alternate_study/publication.yaml")
alternate_report.write_excel(other.directory / "manuscript-tables.xlsx", layout=alternate_layout, requirements=alternate_requirements)
